In [1]:
import numpy as np

In [2]:
np.random.seed(1)

class0 = np.random.randn(100, 2) + [-2, -2]
class1 = np.random.randn(100, 2) + [2, 2]

X = np.vstack((class0, class1))
y = np.array([0]*100 + [1]*100)

In [3]:
idx = np.random.permutation(len(X))
X = X[idx]
y = y[idx]

In [4]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [5]:
class KNN:
    def __init__(self, k):
        self.k = k

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X_test):
        result = []

        for point in X_test:
            distances = []
            for i in range(len(self.X)):
                d = np.sqrt(np.sum((self.X[i] - point) ** 2))
                distances.append((d, self.y[i]))

            distances.sort(key=lambda x: x[0])
            neighbors = distances[:self.k]

            labels = [label for _, label in neighbors]
            prediction = max(set(labels), key=labels.count)
            result.append(prediction)

        return np.array(result)

In [6]:
class SVM:
    def __init__(self, lr=0.001, epochs=1000):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        y = np.where(y == 0, -1, 1)

        self.w = np.zeros(X.shape[1])
        self.b = 0

        for _ in range(self.epochs):
            for i in range(len(X)):
                condition = y[i] * (np.dot(X[i], self.w) + self.b) >= 1

                if condition:
                    self.w -= self.lr * self.w
                else:
                    self.w += self.lr * y[i] * X[i]
                    self.b += self.lr * y[i]

    def predict(self, X):
        output = np.dot(X, self.w) + self.b
        return np.where(output >= 0, 1, 0)

In [7]:
def metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return precision, recall, f1

In [8]:
knn = KNN(k=5)
knn.fit(X_train, y_train)
knn_pred = knn.predict(X_test)


In [9]:
svm = SVM()
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)


In [10]:
kp, kr, kf = metrics(y_test, knn_pred)
sp, sr, sf = metrics(y_test, svm_pred)

print("KNN Results")
print("Precision:", round(kp, 2))
print("Recall   :", round(kr, 2))
print("F1 Score :", round(kf, 2))

print("\nSVM Results")
print("Precision:", round(sp, 2))
print("Recall   :", round(sr, 2))
print("F1 Score :", round(sf, 2))

KNN Results
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

SVM Results
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
